# Prototype of an Asymptotic Numerical Continuation Method for Parametrization

In [ ]:
import os
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer, param_utils
import parametrization, benchmark
import numpy as np

# m = mesh.Mesh('../models/cow2Disc.msh')
m = param_utils.load('../models/lucy.msh.xz')

In [ ]:
# Scale so that the surface area is pi (as in [Su et al. 2020])
m.setVertices(m.vertices() * np.sqrt(np.pi / m.volume))

In [ ]:
uv = mesh_energy.NodalVars(m, 2)
uv_init = param_utils.tutteInitialization(m)
uv.setVars(uv_init.ravel())

In [ ]:
# em = MeshFEM.EmbeddedMesh(m, uv)
# v = viewer.Viewer(em, wireframe=True)
# v.show()

In [ ]:
import continuation_parametrization
param = continuation_parametrization.symmetric_dirichlet_param(m, uv)
objectives = [param]

In [ ]:
# Construct parametrization energy and problem
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, objectives)
opt = prob.optimizer()

In [ ]:
import flip_avoiding_step_length
prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())
prob.initialFeasibleStepLengthComputer.backoffFactor = 0.95

In [ ]:
prob.hessianShift = 1e-9
prob.useRelativeHessianShift = True

In [ ]:
prob.objective()

In [ ]:
DEGREE = 3

In [ ]:
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 2
opt.options.hessianProjectionController.startWithProjectionActive = False

In [ ]:
# benchmark.reset()
# param.setInterpolatedReference(0, uv_init.ravel())
# opt.options.hessianProjectionController.reset()
# prob.invalidateCachedHessian()
# opt.update_factorizations()
# print(prob.hessianWasProjected)
# benchmark.report()

In [ ]:
benchmark.reset()
dl_tgt = 0.2
l = 0
param.setInterpolatedReference(0, uv_init.ravel())
prob.setVars(uv_init.ravel())

while l < 1.0 - 1e-6:
    prob.invalidateCachedHessian()
    # opt.options.hessianProjectionController.reset()
    opt.update_factorizations()
    c = param.computeTaylorCoefficients(opt.hessian_factorization, DEGREE)
    
    dl = min(1.0 - l, dl_tgt)
    print('dl: ', dl)
    l += dl
    param.setInterpolatedReference(l, uv_init.ravel())
    x0 = prob.getVars()
    min_e = np.inf
    min_e_x = None
    for d in range(len(c) + 1):
        x = x0.copy()
        for j in range(d):
            x += dl**(j + 1) * c[j]
        prob.setVars(x)
        print(prob.energy(), np.linalg.norm(prob.gradient()))
        if (prob.energy() < min_e):
            min_e = prob.energy()
            min_e_x = x
    x = min_e_x
    alpha = 1.0
    if np.isinf(min_e):
        alpha = prob.customFeasibleStepLength(x0, x - x0)
    l += alpha * dl - dl
    print('alpha: ', alpha)
    param.setInterpolatedReference(l, uv_init.ravel())
    prob.setVars(x0 + alpha * (x - x0))

    x0 = prob.getVars()
    opt.options.niter = 1
    opt.options.gradTol = 1e-6
    opt.optimize()
    # print('hessianWasProjected: ', prob.hessianWasProjected)
benchmark.report()

In [ ]:
# TODO:
# Try attenuating the Hessian projection amount (interpolate down to no projection)

# Experiments With Arclen Parametrization and Padé Approximation

Let $C \in \mathbb{R}^{n \times d}$ hold the vector-valued coefficient of term $a^j$ in column $j$.
In other words, $C$ is `c.T` above
Then, employing the QR factorization $C = QR$:
$$
    \sum_j (C {\bf e}_j)a^j  =  \sum_j (Q R {\bf e}_j)a^j = \sum_j \sum_{i \le j} {\bf q}_i r_{ij} a^j
    = \sum_i \sum_{j \ge i} {\bf q}_i r_{ij} a^j
    = \sum_i {\bf q}_i \left(\sum_{j \ge i} r_{ij} a^j\right)
$$
in other words, each row of $R$  in 

We could use another orthonormal basis for the range of $C$, e.g., from the SVD, with the modification that the coefficient matrix $R$ is in general no longer upper triangular.

In [ ]:
param.setInterpolatedReference(0, uv_init.ravel())
opt.update_factorizations()
c = param.computeTaylorCoefficients(opt.hessian_factorization, 20)

In [ ]:
benchmark.reset()
c_alen = param.computeTaylorCoefficientsArclen(opt.hessian_factorization, 20)
# benchmark.report()

In [ ]:
np.set_printoptions(linewidth=1000)

In [ ]:
Q, R = np.linalg.qr(np.transpose(c))

In [ ]:
prob.getVars()

In [ ]:
dl = 0.2
l = dl
param.setInterpolatedReference(l, uv_init.ravel())
x0 = uv_init.ravel()
for d in range(len(c) + 1):
    x = x0.copy()
    for j in range(d):
        x += dl**(j + 1) * c[j]
    prob.setVars(x)
    print(prob.energy(), np.linalg.norm(prob.gradient()))

In [ ]:
da = 0.01
for p in range(1, 20):
    Q, R = np.linalg.qr(np.array(c[:p]).T)
    A = R.copy()
    A[-1, -1] = 1
    import scipy
    d = scipy.linalg.solve_triangular(A, np.identity(p)[-1])[::-1]
    Delta = lambda n: np.poly1d(d[:n + 1][::-1])

    x0 = uv_init.ravel()
    den = Delta(p - 1)(da)
    x = x0 + np.sum([c[i - 1] * (da**i * Delta((p - 1) - i)(da) / den) for i in range(1, p + 1)], axis=0)
    # x = x0 + np.sum([c[i - 1] * da**i for i in range(1, p + 1)], axis=0)
    param.setInterpolatedReference(da, x0)
    prob.setVars(x)
    print(prob.energy(), np.linalg.norm(prob.gradient()))

In [ ]:
from matplotlib import pyplot as plt
a = np.linspace(0, 1, 100)
for i in range(1, 6):
    plt.plot(a,  Delta(i)(a))

In [ ]:
a0 = 0
da = 1.9
for p in range(1, 20 + 1):
    Q, R = np.linalg.qr(np.array(c_alen[0][:p]).T)
    A = R.copy()
    A[-1, -1] = 1
    # A += 0.1 * np.random.normal(size=(p,p))
    import scipy
    d = scipy.linalg.solve_triangular(A, np.identity(p)[-1])[::-1]

    x0 = uv_init.ravel()
    den = Delta(p - 1)(da)
    x = x0 + np.sum([c_alen[0][i - 1] * (da**i * Delta((p - 1) - i)(da) / den) for i in range(1, p + 1)], axis=0)
    l = a0 + np.sum([c_alen[1][i - 1] * (da**i * Delta((p - 1) - i)(da) / den) for i in range(1, p + 1)], axis=0)
    # x = x0 + np.sum([c[i - 1] * da**i for i in range(1, p + 1)], axis=0)
    param.setInterpolatedReference(l, x0)
    prob.setVars(x)
    print(l, prob.energy(), np.linalg.norm(prob.gradient()))